In [1]:
import warnings

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns
from polars import selectors as cs
from ..baseline_v3.module.myload import load_data

warnings.filterwarnings('ignore')
%matplotlib inline

ImportError: attempted relative import with no known parent package

In [ ]:
train,_ = load_data()
train = train.drop(["Id"])
print(f"train load complete\r\nsize {train.shape}")

In [ ]:
corr = train.select(cs.numeric()).to_pandas().corr()
sns.heatmap(corr,xticklabels=True, yticklabels=True)

In [ ]:
corr["SalePrice"].sort_values(ascending=False).head(10)

In [ ]:
sns.scatterplot(data=train,x="GrLivArea",y="SalePrice",hue="OverallQual")

In [ ]:
train = train.filter(
    ~((pl.col("GrLivArea")>4000) & (pl.col("SalePrice")<200000))
    )
train.shape

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# SalePriceとの相関係数トップ3でクラスタリング
kmeans_inputs = train.select([
    "OverallQual",
    "GrLivArea",
    "GarageArea",
    "SalePrice"])
scaler = StandardScaler()

# クラスタリング実行
cluster_size = 7
standardized_mat = scaler.fit_transform(kmeans_inputs.to_pandas())
kmeans = KMeans(n_clusters=cluster_size,random_state=42)
clustering_df = kmeans_inputs.with_columns(cluster = kmeans.fit_predict(standardized_mat))

# クラスタリング結果のサマリdfを作成
clustering_df_agg = clustering_df.group_by("cluster").agg(
    pl.len().alias("count"),
    pl.col("SalePrice").mean().alias("mean"),
    pl.col("SalePrice").median().alias("median"),
    pl.col("SalePrice").min().alias("min"),
    pl.col("SalePrice").max().alias("max"),
    ).with_columns(
        width = pl.col("max") - pl.col("min"),
        rank = pl.col("mean").rank(descending=True).cast(pl.Int8)
    )
    
# クラスタ毎のSalePriceランク列を付与
clustering_df = clustering_df.join(
    clustering_df_agg.select(["cluster","rank"]),
    "cluster",
    "left")

# 結果確認
pl.Config.set_tbl_rows(-1)
display(clustering_df_agg.sort("rank"))
pl.Config.restore_defaults()
sns.pairplot(
    data=clustering_df.select(cs.exclude("cluster")).to_pandas(),
    hue="rank",palette="pastel")